# PSELDNets — Colab セットアップ

## ⚠️ 使用前に必ず確認

1. **ランタイム → ランタイムを切断して削除** で完全にリセット
2. **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択
3. セルを **上から順に** 実行（途中でスキップしない）
4. **再起動は不要です**

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU未接続。ランタイムのタイプを T4 GPU に変更してください。'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既にあります: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')
!ls

## 3. パッケージインストール

`numpy` / `h5py` / `scipy` / `torch` は Colab に最初から入っています。  
ここでは**触らず**、不足しているものだけ追加します（再起動不要）。

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

print('完了')

## 4. 動作確認

In [ ]:
import numpy as np, h5py, lightning, torchmetrics, librosa
print(f'numpy       : {np.__version__}')
print(f'h5py        : {h5py.__version__}')
print(f'lightning   : {lightning.__version__}')
print(f'torchmetrics: {torchmetrics.__version__}')
print(f'librosa     : {librosa.__version__}')
print('すべて OK')

---
## 5. チェックポイントのダウンロード

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    print('ダウンロード中...')
    src = hf_hub_download(
        repo_id='Jinbo-HU/PSELDNets',
        filename='model/mACCDOA-HTSAT-0.567.ckpt',
        repo_type='dataset',
    )
    shutil.copy(src, CKPT)

print(f'OK: {CKPT}  ({os.path.getsize(CKPT)/1e6:.0f} MB)')

## 6. クラス定義ファイル (TSV)

In [ ]:
os.makedirs('datasets', exist_ok=True)

for tsv in ['cls_indices_train.tsv', 'cls_indices_test.tsv']:
    dest = f'datasets/{tsv}'
    if not os.path.exists(dest):
        src = hf_hub_download(
            repo_id='Jinbo-HU/PSELDNets',
            filename=f'dataset/{tsv}',
            repo_type='dataset',
        )
        shutil.copy(src, dest)
    print(f'OK: {dest}')

## 7. テストデータのダウンロード（約4GB）

In [ ]:
import zipfile

ZIP = 'datasets/test360_ov3.zip'
DATA_DIR = 'datasets/test360_ov3'

if not os.path.exists(f'{DATA_DIR}/foa'):
    if not os.path.exists(ZIP):
        print('ダウンロード中（約4GB）...')
        src = hf_hub_download(
            repo_id='Jinbo-HU/PSELDNets',
            filename='dataset/test360_ov3.zip',
            repo_type='dataset',
        )
        shutil.copy(src, ZIP)
        print(f'ダウンロード完了: {os.path.getsize(ZIP)/1e9:.2f} GB')

    print('解凍中...')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('datasets/')
    print('解凍完了')
else:
    print(f'既にあります: {DATA_DIR}')

!ls datasets/test360_ov3/

---
## 8. 前処理（FLAC → HDF5 特徴量）

数分かかります。

In [ ]:
if not os.path.exists('_hdf5') or len(os.listdir('_hdf5')) == 0:
    !python src/preproc.py dataset=test360_ov3
else:
    print('既に前処理済み')
    !ls _hdf5/

## 9. 推論実行

In [ ]:
# 事前確認
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt', 'チェックポイント'),
    ('datasets/cls_indices_train.tsv',  'TSV train'),
    ('datasets/cls_indices_test.tsv',   'TSV test'),
    ('datasets/test360_ov3/foa',        'FOA データ'),
    ('_hdf5',                           'HDF5 特徴量'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

In [ ]:
!python src/infer.py \
    experiment=synth_maccdoa \
    ckpt_path=ckpts/mACCDOA-HTSAT-0.567.ckpt \
    model.kwargs.pretrained_path=null

---
# DCASE2021 ファインチューニング

## 10. データダウンロード（FOA のみ、約 7.6 GB）

| ファイル | サイズ | 説明 |
|---------|--------|------|
| foa_dev.zip + foa_dev.z01 | 5.7 GB | 訓練データ（分割 zip） |
| foa_eval.zip | 1.9 GB | 評価データ |
| metadata_dev.zip | 2 MB | 訓練ラベル |
| metadata_eval.zip | 660 kB | 評価ラベル |

In [ ]:
import os

DCASE = 'datasets/DCASE2021'
os.makedirs(DCASE, exist_ok=True)

BASE = 'https://zenodo.org/records/5476980/files'

files = [
    'foa_dev.zip',       # 1.4 GB（分割 zip のメイン部分）
    'foa_dev.z01',       # 4.3 GB（分割 zip のパート 1）
    'foa_eval.zip',      # 1.9 GB
    'metadata_dev.zip',  # 2 MB
    'metadata_eval.zip', # 660 kB
]

for fname in files:
    dest = f'{DCASE}/{fname}'
    if not os.path.exists(dest):
        print(f'ダウンロード中: {fname}')
        !wget -q --show-progress -O {dest} {BASE}/{fname}?download=1
    else:
        print(f'スキップ（既にあります）: {fname}')

print('\nダウンロード完了')
!ls -lh {DCASE}/

## 11. 解凍 + ディレクトリ整理

`foa_dev` は分割 zip（`.zip` + `.z01`）なので、結合してから解凍します。

In [ ]:
DCASE = 'datasets/DCASE2021'

# foa_dev: 分割 zip を結合してから解凍
if not os.path.exists(f'{DCASE}/foa_dev'):
    print('foa_dev を結合・解凍中（数分かかります）...')
    !zip -s 0 {DCASE}/foa_dev.zip --out {DCASE}/foa_dev_agg.zip
    !unzip -q {DCASE}/foa_dev_agg.zip -d {DCASE}/
    !rm {DCASE}/foa_dev_agg.zip
    print('foa_dev 完了')

if not os.path.exists(f'{DCASE}/foa_eval'):
    print('foa_eval を解凍中...')
    !unzip -q {DCASE}/foa_eval.zip -d {DCASE}/

if not os.path.exists(f'{DCASE}/metadata_dev'):
    !unzip -q {DCASE}/metadata_dev.zip -d {DCASE}/

if not os.path.exists(f'{DCASE}/metadata_eval'):
    !unzip -q {DCASE}/metadata_eval.zip -d {DCASE}/

# 解凍後はサブフォルダに wav/csv が入っているので平坦化する
print('ディレクトリ整理中...')
for folder, ext in [('foa_dev', 'wav'), ('foa_eval', 'wav'),
                    ('metadata_dev', 'csv'), ('metadata_eval', 'csv')]:
    path = f'{DCASE}/{folder}'
    !find {path} -mindepth 2 -name "*.{ext}" -exec mv -t {path}/ {{}} + 2>/dev/null || true
    !find {path} -mindepth 1 -maxdepth 1 -type d -exec rm -rf {{}} + 2>/dev/null || true

print('\n整理後のファイル数:')
for d in ['foa_dev', 'foa_eval', 'metadata_dev', 'metadata_eval']:
    n = len(os.listdir(f'{DCASE}/{d}')) if os.path.exists(f'{DCASE}/{d}') else 0
    print(f'  {d}/: {n} ファイル')

## 12. 前処理

In [ ]:
# 訓練データの前処理
!python src/preproc.py dataset=DCASE2021 wav_format=.wav

# 評価データの前処理
!python src/preproc.py dataset=DCASE2021 dataset_type=eval wav_format=.wav

## 13. ファインチューニング（約 70 エポック）

In [ ]:
# メモリ不足時はバッチサイズを下げる
!python src/train.py experiment=dcase2021/finetune_maccdoa_augmix1 model.batch_size=8